# 🚄 Bangkok Rail Transit: Altair AI Studio Data Cleaning & ML Crowd Prediction
### From Raw Data Cleaning (Altair RapidMiner Pipeline) to Machine Learning Forecasting
**Course:** Data Mining & Warehouse / Bangkok Transit Project  
**Workflow Overview:**
1. **Part I: Altair AI Studio (RapidMiner) Pipeline in Python**
   - Recreate the exact operators from `Data_Cleaning.rmp`:
     - Clean `Number_Passenger`: remove commas, replace non-numeric characters with `"0"`, parse to numeric.
     - Replace `Note`: translate Thai earthquake alert to English.
     - Extract `Year`, `Month`, and create `MonthYear` temporal attribute.
     - Turbo Prep Monthly Aggregate Pivot Table: `sum(Number_Passenger)` grouped by `MonthYear` and `Organization`.
     - Output clean dataset (`DataProcessed.csv`).
2. **Part II: Exploratory Data Analysis & Transit Insights**
   - Statistical distribution across 9 operators, weekday vs weekend commuter drops, 4-year trend analysis.
3. **Part III: Feature Engineering (22 Tabular Features)**
   - Calendar, cyclical encodings, Thai public holidays (Songkran, New Year, etc.), historical lags ($	ext{lag}_7, 	ext{lag}_{14}, 	ext{lag}_{364}$), and rolling windows.
4. **Part IV: Model Training & Evaluation (HistGradientBoosting / LightGBM)**
   - Chronological 80/20 train/test split, MAE, RMSE, MAPE %, and $R^2$ validation.
5. **Part V: Crowd Density Normalization & Interactive Prediction**
   - Convert passenger forecasts into 0–100% Crowd Density Index and qualitative levels (Light, Moderate, Busy, Crowded).
   - Whole-day aggregate notice vs morning/evening rush hours.
6. **Part VI: Export Artifacts for Web/API Integration**



## 1. Setup & Dependencies
Install necessary packages (`holidays` for Thai holiday calendar, `scikit-learn`, `seaborn`, `joblib`).


In [ ]:
# Install required packages
!pip install holidays scikit-learn pandas numpy matplotlib seaborn joblib -q

import os
import re
import json
from datetime import datetime, date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("✅ Setup complete! Python libraries imported successfully.")



### Upload Raw `daily_passenger.csv`
If running on Google Colab, upload the original raw `daily_passenger.csv` file using the prompt below.


In [ ]:
# Colab file upload helper
RAW_DATASET_PATH = 'daily_passenger.csv'

if not os.path.exists(RAW_DATASET_PATH):
    try:
        from google.colab import files
        print("Please upload original raw 'daily_passenger.csv':")
        uploaded = files.upload()
        for filename in uploaded.keys():
            RAW_DATASET_PATH = filename
    except ImportError:
        # Local workspace fallbacks
        if os.path.exists('data/daily_passenger_raw.csv'):
            RAW_DATASET_PATH = 'data/daily_passenger_raw.csv'
        elif os.path.exists('data/daily_passenger.csv'):
            RAW_DATASET_PATH = 'data/daily_passenger.csv'
        elif os.path.exists('../data/daily_passenger.csv'):
            RAW_DATASET_PATH = '../data/daily_passenger.csv'

print(f"Using raw dataset: {RAW_DATASET_PATH}")



## 2. Altair AI Studio (RapidMiner) Pipeline in Python
Here we recreate the exact sequence of operators defined in **`Data_Cleaning.rmp`**:

```
[Retrieve daily_passenger]
       │
       ▼
[Clean Number_Passenger]
   ├── Replace "," (remove commas)
   ├── Generate Attributes (regex: [^0-9] -> "0")
   └── Parse Numbers (convert to numeric)
       │
       ▼
[Replace Notes] ("ปิดให้บริการชั่วคราว..." -> "Temporarily closed due to an earthquake")
       │
       ▼
[Generate Attributes (2)] (Year, Month, MonthYear = Year + "-" + Month)
       │
       ▼
[Turbo Prep Monthly Pivot] (Group by MonthYear, sum(Number_Passenger) by Organization)
       │
       ▼
[Select & Order Attributes] (No. | Date | Organization | Number_Passenger | Note)
```



In [ ]:
# Step 2.1: Inspect Raw Dataset
df_raw = pd.read_csv(RAW_DATASET_PATH)
print(f"Raw shape: {df_raw.shape} (Rows x Columns)")
print("\nColumn Data Types:")
print(df_raw.dtypes)
print("\nFirst 5 Raw Records (Notice comma strings in Number_Passenger):")
display(df_raw.head())



In [ ]:
# Step 2.2: Recreate 'Clean Number_Passenger' Subprocess
# RapidMiner: Replace ',' -> Generate Attributes '[^0-9]' -> Parse Numbers
df_cleaned = df_raw.copy()

# 1. Remove commas
df_cleaned['Number_Passenger'] = df_cleaned['Number_Passenger'].astype(str).str.replace(',', '', regex=False)

# 2. Trim and replace any non-digit character with '0'
df_cleaned['Number_Passenger'] = df_cleaned['Number_Passenger'].str.strip().str.replace(r'[^0-9.]', '0', regex=True)

# 3. Convert to numeric (float64)
df_cleaned['Number_Passenger'] = pd.to_numeric(df_cleaned['Number_Passenger'], errors='coerce').fillna(0)

# Step 2.3: Recreate 'Replace Notes' Operator
# RapidMiner: Replace 'ปิดให้บริการชั่วคราว เนื่องจากเกิดเหตุแผ่นดินไหว' -> 'Temporarily closed due to an earthquake'
thai_closed_note = "ปิดให้บริการชั่วคราว เนื่องจากเกิดเหตุแผ่นดินไหว"
eng_closed_note = "Temporarily closed due to an earthquake"

df_cleaned['Note'] = df_cleaned['Note'].astype(str).replace({
    thai_closed_note: eng_closed_note,
    'nan': np.nan,
    'None': np.nan
})

# Display affected note rows
note_matches = df_cleaned[df_cleaned['Note'].notna() & (df_cleaned['Note'] != 'nan')]
print(f"✅ Cleaned Number_Passenger and Notes. Sample note rows ({len(note_matches)} entries):")
display(note_matches.head())



In [ ]:
# Step 2.4: Recreate 'Generate Attributes (2)' - Month, Year, MonthYear
# RapidMiner regex:
#   Year = replaceAll([Date], '^.*?(\\d{4}).*$', '$1')
#   Month = replaceAll([Date], '^\\d{1,2}/(\\d{1,2})/.*$', '$1')
#   MonthYear = concat([Year], '-', [Month])

# Parse Date object
df_cleaned['Date_Parsed'] = pd.to_datetime(df_cleaned['Date'], format='%d/%m/%Y')
df_cleaned['Year'] = df_cleaned['Date_Parsed'].dt.year.astype(str)
df_cleaned['Month'] = df_cleaned['Date_Parsed'].dt.month.astype(str)
df_cleaned['MonthYear'] = df_cleaned['Year'] + '-' + df_cleaned['Month'].str.zfill(2)

# Step 2.5: Recreate 'Reorder Attributes'
# RapidMiner: No. | Date | Organization | Number_Passenger | Note | MonthYear
attribute_order = ['No.', 'Date', 'Organization', 'Number_Passenger', 'Note', 'MonthYear']
df_cleaned = df_cleaned[attribute_order].sort_values(by=['No.']).reset_index(drop=True)

print("✅ Attributes ordered according to Altair RapidMiner specification:")
display(df_cleaned.head())



In [ ]:
# Step 2.6: Recreate Altair 'Turbo Prep Process - Monthly Passenger Pivot Table'
# RapidMiner: Group by MonthYear, columns = Organization, values = sum(Number_Passenger)
# Column ordering: SRT, ARL, Red Line, MRT Blue Line, MRT Purple Line, BTS Green Line, BTS Gold Line, MRT Yellow Line, MRT Pink Line

monthly_pivot = df_cleaned.pivot_table(
    index='MonthYear',
    columns='Organization',
    values='Number_Passenger',
    aggfunc='sum'
).fillna(0)

rapidminer_col_order = [
    'SRT', 'ARL', 'Red Line', 'MRT Blue Line', 'MRT Purple Line',
    'BTS Green Line', 'BTS Gold Line', 'MRT Yellow Line', 'MRT Pink Line'
]
existing_cols = [c for c in rapidminer_col_order if c in monthly_pivot.columns]
monthly_pivot = monthly_pivot[existing_cols]

print("📊 Monthly Passenger Aggregate Pivot Table (RapidMiner Turbo Prep Output):")
display(monthly_pivot.head(10))

# Save the cleaned dataset as DataProcessed.csv
df_cleaned.to_csv("DataProcessed.csv", index=False)
print("💾 Cleaned dataset exported as 'DataProcessed.csv'")



In [ ]:
# Visualize Monthly Passenger Volume Trends (Altair Process Analysis)
fig, ax = plt.subplots(figsize=(15, 6))
for col in ['BTS Green Line', 'MRT Blue Line', 'ARL', 'MRT Purple Line', 'Red Line', 'MRT Yellow Line', 'MRT Pink Line']:
    if col in monthly_pivot.columns:
        ax.plot(monthly_pivot.index, monthly_pivot[col] / 1e6, marker='o', markersize=4, label=col, linewidth=1.8)

ax.set_title("Monthly Bangkok Rail Passenger Volume by Operator (RapidMiner Aggregate)", fontsize=14, fontweight='bold')
ax.set_ylabel("Total Monthly Passengers (Millions)")
ax.set_xlabel("Month-Year")
ax.legend(loc='upper left', frameon=True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



## 3. Transition to Daily Machine Learning Pipeline
Now we model **daily transit ridership and crowd density** across Bangkok's railway corridors.
We standardize operator line IDs and construct a daily time-series matrix (1,461 days $\times$ 9 lines).


In [ ]:
LINE_MAPPING = {
    "BTS Green Line": "BTS_Green",
    "BTS Gold Line": "BTS_Gold",
    "MRT Blue Line": "MRT_Blue",
    "MRT Purple Line": "MRT_Purple",
    "MRT Yellow Line": "MRT_Yellow",
    "MRT Pink Line": "MRT_Pink",
    "ARL": "ARL",
    "Red Line": "SRT_Red",
    "SRT": "SRT",
}

df_ml = df_cleaned.copy()
df_ml['LineId'] = df_ml['Organization'].map(LINE_MAPPING).fillna(df_ml['Organization'])
df_ml['Date'] = pd.to_datetime(df_ml['Date'], format='%d/%m/%Y')

# Pivot into daily continuous time-series
df_pivot = df_ml.pivot(index='Date', columns='LineId', values='Number_Passenger').asfreq('D').fillna(0)

print(f"Pivoted Daily Time-Series: {df_pivot.shape} (1461 calendar days x 9 lines)")
display(df_pivot.head(5))



## 4. Exploratory Data Analysis (EDA)
Inspect distribution, 95th percentile capacity ($P_{95}$), and weekday rush hour behavior.


In [ ]:
# Statistical summary & P95 capacity benchmarks
stats = df_pivot.describe().round(0).T[['count', 'mean', 'std', 'min', '50%', '75%', 'max']]
stats['P95 Capacity'] = [int(np.percentile(df_pivot[c][df_pivot[c] > 0], 95)) for c in stats.index]
display(stats)



In [ ]:
# Weekday Commuter Rush vs Weekend Leisure Travel
df_comp = df_pivot.copy()
df_comp['DayType'] = np.where(df_comp.index.dayofweek >= 5, 'Weekend', 'Weekday')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, line in enumerate(['BTS_Green', 'MRT_Blue', 'ARL']):
    sns.boxplot(data=df_comp, x='DayType', y=line, ax=axes[i], palette=['#38bdf8', '#fb7185'])
    axes[i].set_title(f"{line}: Weekday vs Weekend", fontweight='bold')
    axes[i].set_ylabel("Passengers/day")
    axes[i].set_xlabel("")

plt.tight_layout()
plt.show()



## 5. Feature Engineering
Construct **22 features** capturing:
1. **Calendar & Cyclical:** Day of week, Month, Day of year, Quarter, `is_weekend`, cyclical sine/cosine encodings
2. **Thai Public Holidays & Long Weekends:** Songkran, New Year, King Rama X's Birthday, substitution days, school break periods
3. **Historical Memory (Lags & Rolling Windows):** $\text{lag}_7$ (same day last week), $\text{lag}_{14}$, $\text{lag}_{364}$ (same day last year), 7-day, 14-day, 28-day moving averages and standard deviation



In [ ]:
THAI_HOLIDAYS = {
    "01-01": {"en": "New Year's Day", "th": "วันขึ้นปีใหม่"},
    "01-02": {"en": "New Year Holiday", "th": "วันหยุดชดเชยปีใหม่"},
    "02-26": {"en": "Makha Bucha Day", "th": "วันมาฆบูชา"},
    "04-06": {"en": "Chakri Memorial Day", "th": "วันจักรี"},
    "04-13": {"en": "Songkran Festival", "th": "วันสงกรานต์"},
    "04-14": {"en": "Songkran Festival", "th": "วันสงกรานต์"},
    "04-15": {"en": "Songkran Festival", "th": "วันสงกรานต์"},
    "04-16": {"en": "Songkran Substitution", "th": "วันหยุดชดเชยสงกรานต์"},
    "05-01": {"en": "National Labor Day", "th": "วันแรงงานแห่งชาติ"},
    "05-04": {"en": "Coronation Day", "th": "วันฉัตรมงคล"},
    "06-03": {"en": "Queen's Birthday", "th": "วันเฉลิมพระชนมพรรษาสมเด็จพระบรมราชินี"},
    "07-28": {"en": "King Rama X's Birthday", "th": "วันเฉลิมพระชนมพรรษา ร.10"},
    "08-12": {"en": "Mother's Day", "th": "วันแม่แห่งชาติ"},
    "10-13": {"en": "King Rama IX Memorial Day", "th": "วันนวมินทรมหาราช"},
    "10-23": {"en": "Chulalongkorn Day", "th": "วันปิยมหาราช"},
    "12-05": {"en": "National Day / Father's Day", "th": "วันชาติ / วันพ่อแห่งชาติ"},
    "12-10": {"en": "Constitution Day", "th": "วันรัฐธรรมนูญ"},
    "12-31": {"en": "New Year's Eve", "th": "วันสิ้นปี"},
}

def extract_calendar_features(dates: pd.DatetimeIndex) -> pd.DataFrame:
    df = pd.DataFrame(index=dates)
    df['day_of_week'] = dates.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['month'] = dates.month
    df['day_of_month'] = dates.day
    df['day_of_year'] = dates.dayofyear
    df['quarter'] = dates.quarter
    df['week_of_year'] = dates.isocalendar().week.astype(int)

    # Cyclical encodings
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)

    # Thai Holiday indicators
    is_holiday, is_songkran, is_new_year = [], [], []
    for d in dates:
        key = f"{d.month:02d}-{d.day:02d}"
        h = THAI_HOLIDAYS.get(key)
        is_holiday.append(1 if h else 0)
        is_songkran.append(1 if h and "Songkran" in h["en"] else 0)
        is_new_year.append(1 if h and "New Year" in h["en"] else 0)

    df['is_holiday'] = is_holiday
    df['is_songkran'] = is_songkran
    df['is_new_year'] = is_new_year
    df['is_school_break'] = df['month'].isin([3, 4, 10]).astype(int)
    return df

df_calendar = extract_calendar_features(df_pivot.index)

def build_line_dataset(line_name: str, pivoted_df: pd.DataFrame, cal_df: pd.DataFrame):
    s = pivoted_df[line_name].copy()
    active_idx = (s > 0).idxmax() if (s > 0).any() else s.index[0]
    s_active = s.loc[active_idx:]

    X = cal_df.loc[s_active.index].copy()
    X['lag_7'] = s.shift(7).loc[s_active.index]
    X['lag_14'] = s.shift(14).loc[s_active.index]
    X['lag_364'] = s.shift(364).loc[s_active.index]

    X['rolling_mean_7'] = s.shift(1).rolling(7).mean().loc[s_active.index]
    X['rolling_mean_14'] = s.shift(1).rolling(14).mean().loc[s_active.index]
    X['rolling_mean_28'] = s.shift(1).rolling(28).mean().loc[s_active.index]
    X['rolling_std_7'] = s.shift(1).rolling(7).std().loc[s_active.index]

    valid_mask = X['rolling_mean_28'].notna() & X['lag_7'].notna()
    return X.loc[valid_mask], s_active.loc[valid_mask]

print("✅ Feature engineering pipeline ready!")



## 6. Model Training & Time-Series Evaluation
Using **Histogram-Based Gradient Boosting (`HistGradientBoostingRegressor`)** with a chronological **80% train / 20% test holdout split**.


In [ ]:
results = []
trained_models = {}
line_metadata = {}
predictions_dict = {}

calendar_feature_cols = [c for c in df_calendar.columns]

print(f"{'Line':<14} | {'Train N':<7} | {'Test N':<6} | {'MAE':<10} | {'MAPE %':<8} | {'R2 Score':<8}")
print("-" * 65)

for line in df_pivot.columns:
    X, y = build_line_dataset(line, df_pivot, df_calendar)
    
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    # Primary model (Calendar + Lags)
    model = HistGradientBoostingRegressor(
        max_iter=150, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=10, random_state=42
    )
    model.fit(X_train, y_train)

    # Calendar-only model (Fallback for arbitrary future dates)
    cal_model = HistGradientBoostingRegressor(
        max_iter=150, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=10, random_state=42
    )
    cal_model.fit(X_train[calendar_feature_cols], y_train)

    # Holdout test predictions
    y_pred = model.predict(X_test)
    predictions_dict[line] = (y_test, y_pred)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / np.maximum(y_test, 1))) * 100
    r2 = r2_score(y_test, y_pred)

    results.append({
        'Line': line, 'MAE': round(mae), 'RMSE': round(rmse), 'MAPE %': round(mape, 2), 'R2': round(r2, 3)
    })
    print(f"{line:<14} | {len(X_train):<7} | {len(X_test):<6} | {mae:<10.0f} | {mape:<7.2f}% | {r2:<8.3f}")

    # Retrain on 100% data for production deployment
    model_prod = HistGradientBoostingRegressor(
        max_iter=180, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=10, random_state=42
    ).fit(X, y)
    cal_prod = HistGradientBoostingRegressor(
        max_iter=180, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=10, random_state=42
    ).fit(X[calendar_feature_cols], y)

    s_active = df_pivot[line][df_pivot[line] > 0]
    p95_capacity = int(np.percentile(s_active, 95))

    trained_models[line] = {
        'full_model': model_prod,
        'cal_model': cal_prod,
        'features': list(X.columns),
        'cal_features': calendar_feature_cols
    }
    line_metadata[line] = {
        'p95_capacity': p95_capacity,
        'max_capacity': int(s_active.max()),
        'mean_ridership': int(s_active.mean())
    }

df_eval = pd.DataFrame(results).sort_values(by='R2', ascending=False)
display(df_eval)



## 7. Holdout Test Set Plots (Actual vs Predicted)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)
for i, line in enumerate(['BTS_Green', 'MRT_Blue', 'ARL']):
    y_test, y_pred = predictions_dict[line]
    axes[i].plot(y_test.index, y_test.values, label='Actual Historical Ridership', color='#0284c7', linewidth=1.5)
    axes[i].plot(y_test.index, y_pred, label='ML Model Predicted', color='#f43f5e', linestyle='--', linewidth=1.5)
    axes[i].set_title(f"{line} — Holdout Test Set (Actual vs Predicted)", fontweight='bold')
    axes[i].set_ylabel("Daily Passengers")
    axes[i].legend(loc='upper right')

plt.xlabel("Date")
plt.tight_layout()
plt.show()



## 8. Interactive Prediction Tool
Forecast crowd density and ridership for any chosen travel date (e.g. Songkran festival vs regular Monday).
Includes the whole-day aggregate notice explaining rush hour variance.


In [ ]:
def calculate_crowd_level(passenger_count: float, p95_cap: float):
    density_pct = min(100, max(10, round((passenger_count / p95_cap) * 100)))
    if density_pct >= 85:
        return density_pct, "Crowded (หนาแน่นสูง)", "#ef4444"
    elif density_pct >= 70:
        return density_pct, "Busy (คึกคัก)", "#f97316"
    elif density_pct >= 50:
        return density_pct, "Moderate (ปานกลาง)", "#eab308"
    else:
        return density_pct, "Light (เบาบาง)", "#10b981"

def predict_transit_crowd(target_date_str: str):
    target_dt = pd.to_datetime(target_date_str)
    cal_features = extract_calendar_features(pd.DatetimeIndex([target_dt]))
    holiday = THAI_HOLIDAYS.get(f"{target_dt.month:02d}-{target_dt.day:02d}")
    
    day_name = target_dt.day_name()
    print("=" * 75)
    print(f"🔮 Bangkok Rail Transit Crowd Forecast: {target_date_str} ({day_name})")
    if holiday:
        print(f"🌟 Thai Public Holiday: {holiday['en']} ({holiday['th']})")
    print("=" * 75)
    print(f"{'Line':<14} | {'Predicted Passengers':<20} | {'Crowd Score':<12} | {'Level'}")
    print("-" * 75)

    for line, entry in trained_models.items():
        p95 = line_metadata[line]['p95_capacity']
        try:
            prev_s = df_pivot[line]
            l7 = float(prev_s.loc[target_dt - pd.Timedelta(days=7)])
            l14 = float(prev_s.loc[target_dt - pd.Timedelta(days=14)])
            l364 = float(prev_s.loc[target_dt - pd.Timedelta(days=364)]) if (target_dt - pd.Timedelta(days=364)) in prev_s.index else l7

            X_in = cal_features.copy()
            X_in['lag_7'] = l7
            X_in['lag_14'] = l14
            X_in['lag_364'] = l364
            prev_slice = prev_s.loc[:target_dt - pd.Timedelta(days=1)].tail(28)
            X_in['rolling_mean_7'] = float(prev_slice.tail(7).mean())
            X_in['rolling_mean_14'] = float(prev_slice.tail(14).mean())
            X_in['rolling_mean_28'] = float(prev_slice.mean())
            X_in['rolling_std_7'] = float(prev_slice.tail(7).std() or 0)
            pred = entry['full_model'].predict(X_in[entry['features']])[0]
        except Exception:
            pred = entry['cal_model'].predict(cal_features[entry['cal_features']])[0]

        pred = max(0, round(pred))
        pct, label, color = calculate_crowd_level(pred, p95)
        print(f"{line:<14} | {pred:<20,d} | {pct:>3d}% ({color}) | {label}")

    print("-" * 75)
    print("ℹ️ Note: Predictions reflect full-day aggregate volumes. Peak rush hours (07:00-09:00 & 17:30-19:30)")
    print("   will experience significantly higher platform and train density than quiet off-peak periods.\n")

# Run test forecasts
predict_transit_crowd("2026-11-02") # Regular Monday commute
predict_transit_crowd("2026-04-13") # Songkran festival



## 9. Save & Export Model Artifacts
Export `passenger_models.joblib`, `metadata.json`, and `historical_cache.csv` for the FastAPI backend and React frontend.


In [ ]:
# Export model bundle
os.makedirs("models", exist_ok=True)
joblib.dump(trained_models, "models/passenger_models.joblib")

with open("models/metadata.json", "w", encoding="utf-8") as f:
    json.dump({
        "lines": line_metadata,
        "evaluation": df_eval.to_dict(orient="records")
    }, f, indent=2, ensure_ascii=False)

df_pivot.to_csv("models/historical_cache.csv")
print("✅ Artifacts saved to 'models/' directory:")
print("   - models/passenger_models.joblib")
print("   - models/metadata.json")
print("   - models/historical_cache.csv")

# Trigger browser download when running on Google Colab
try:
    from google.colab import files
    print("\n📥 Downloading trained model bundle to your computer...")
    files.download("models/passenger_models.joblib")
    files.download("models/metadata.json")
except ImportError:
    pass

